In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, Dataset, DataLoader, random_split
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence
)
import os
import random
from copy import deepcopy

In [2]:
def make_deterministic(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

    rng = np.random.default_rng(42)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterinistic = True
        torch.backends.cudnn.benchmark = False

        torch.use_deterministic_algorithms(True, warn_only=True)

make_deterministic(seed=42)

## Przygotowanie danych

In [3]:
device = torch.device('cuda') # torch.device('cpu')
device

device(type='cuda')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!unzip -n /content/drive/MyDrive/sroda_FijalkowskiFilip_ZukowskaRadoslawa/mini_projekt_5/pakiet.zip -d /content/

Archive:  /content/drive/MyDrive/sroda_FijalkowskiFilip_ZukowskaRadoslawa/mini_projekt_5/pakiet.zip
   creating: /content/pakiet/
  inflating: /content/__MACOSX/._pakiet  
  inflating: /content/pakiet/train.pkl  
  inflating: /content/__MACOSX/pakiet/._train.pkl  
  inflating: /content/pakiet/.DS_Store  
  inflating: /content/__MACOSX/pakiet/._.DS_Store  
  inflating: /content/pakiet/test_no_target.pkl  
  inflating: /content/__MACOSX/pakiet/._test_no_target.pkl  
  inflating: /content/pakiet/treść_zadania.txt  
  inflating: /content/__MACOSX/pakiet/._treść_zadania.txt  


In [6]:
import pickle

# Load train data
with open('/content/pakiet/train.pkl', 'rb') as f:
    train_data = pickle.load(f)

# Load test data
with open('/content/pakiet/test_no_target.pkl', 'rb') as f:
    test_data = pickle.load(f)

print(type(train_data))
print(type(test_data))

<class 'list'>
<class 'list'>


In [7]:
train_data[0]

(array([ -1.,  -1.,  -1., ...,  78.,  40., 144.]), 0)

In [8]:
test_data[0]

array([  0.,   0.,   0.,   0.,  88.,  88.,  88.,  88.,  69., 145., 145.,
       145.,   5.,   5.,   5.,   5.,  92.,  88.,  88.,  80.,  45.,  39.,
       190., 190., 124., 112., 112., 112.,  77.,  36.,  36.,  36.,  20.,
        20.,  13., 158., 127.,  55.,  55., 113., 127., 151.,  22., 116.,
       144., 144.,  34.,  68.,  33.,  33.,  33.,  41.,  41.,  47.,  47.,
       180.,  12., 119.,  37., 100., 112., 112.,   2.,  36.,   0.,   0.,
         0.,   0.,  88.,  88., 120., 120.,  12.,   5.,  13.,  13.,  92.,
       149.,  32.,   4.,  12.,  37.,  12., 159.,  13.,   6., 124.,  37.,
       124.,  37.,  37.,   8.,   8., 119.,  78.,  73.,  12.,   0.,   0.,
         0.,  88.,  88.,  88.,  88.,  69., 145., 145., 145.,   5.,   5.,
         5.,   5.,  92.,  88.,  88.,  80.,  45.,  39., 190., 190., 124.,
       112., 112., 112.,  77.,  36.,  36.,  36.,  20.,  20.,  13., 158.,
       127.,  55.,  55., 113., 127., 151.,  22., 116., 144., 144.,  34.,
        68.,  33.,  33.,  33.,  41.,  41.,  47.,  4

### Sprawdzenie rozkładu

In [9]:
labels = [label for _, label in train_data]
df_labels = pd.DataFrame(labels, columns=['class'])
print(df_labels['class'].value_counts(normalize=True).sort_index()*100)

class
0    55.461041
1    16.264035
2     5.239878
3    15.005104
4     8.029942
Name: proportion, dtype: float64


In [10]:
counts = df_labels['class'].value_counts().sort_index()
class_weights = 1 / counts
class_weights = class_weights / class_weights.sum()
print(class_weights)
class_weights = torch.tensor(class_weights.values, dtype=torch.float32)

class
0    0.039066
1    0.133218
2    0.413496
3    0.144395
4    0.269824
Name: count, dtype: float64


# Klasyfikacja serii

In [11]:
len(train_data)

2939

In [12]:
max_val = max(torch.max(seq).item() for seq in train_data) if isinstance(train_data[0], torch.Tensor) else max(max(seq) for seq, _ in train_data)
min_val = min(torch.min(seq).item() for seq in train_data) if isinstance(train_data[0], torch.Tensor) else min(min(seq) for seq, _ in train_data)

print(f"Minimum note value in data: {min_val}")
print(f"Maximum note value in data: {max_val}")

Minimum note value in data: -1.0
Maximum note value in data: 191.0


## Dataset

In [13]:
class TrainDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        sequence, label = self.data[idx]

        sequence = torch.tensor(
            sequence,
            dtype=torch.long
        ) + 1

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return sequence, label


class TestDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        sequence = torch.tensor(
            self.data[idx],
            dtype=torch.long
        ) + 1

        return sequence

## Collate functions

In [14]:
def train_collate_fn(batch):

    sequences = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    lengths = torch.tensor(
        [len(seq) for seq in sequences],
        dtype=torch.long
    )

    padded_sequences = pad_sequence(
        sequences,
        batch_first=True
    )

    labels = torch.stack(labels)

    return padded_sequences, lengths, labels


def test_collate_fn(batch):

    lengths = torch.tensor(
        [len(seq) for seq in batch],
        dtype=torch.long
    )

    padded_sequences = pad_sequence(
        batch,
        batch_first=True
    )

    return padded_sequences, lengths

In [15]:
full_train_dataset = TrainDataset(train_data)

In [16]:
all_labels = [label for _, label in train_data]
train_idx, valid_idx = train_test_split(
    range(len(full_train_dataset)),
    test_size=0.2,
    stratify=all_labels,
    random_state=42
)

train_subset = Subset(full_train_dataset, train_idx)
valid_subset = Subset(full_train_dataset, valid_idx)

In [17]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_collate_fn,
    drop_last=False
)

valid_loader = DataLoader(
    valid_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=train_collate_fn,
    drop_last=False
)

test_dataset = TestDataset(test_data)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=test_collate_fn,
    drop_last=False
)

In [18]:
print(f"Total train samples: {len(train_subset)}")
print(f"Total valid samples: {len(valid_subset)}")
print(f"Total test samples: {len(test_dataset)}")

Total train samples: 2351
Total valid samples: 588
Total test samples: 1103


In [19]:
for seqs, lengths, labels in train_loader:
    print(f"Padded sequences shape: {seqs.shape}") # [batch_size, max_seq_len] or [batch_size, max_seq_len, features]
    print(f"Sequence lengths shape: {lengths.shape}") # [batch_size]
    print(f"Labels shape: {labels.shape}")          # [batch_size]
    break

Padded sequences shape: torch.Size([32, 1696])
Sequence lengths shape: torch.Size([32])
Labels shape: torch.Size([32])


## Klasyfikator

In [22]:
class LSTMClassifier(nn.Module):

    def __init__(self, vocab_size=256, embedding_dim=64, hidden_size=128, num_layers=2, out_size=5, bidirectional = False):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)

        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size * self.bidirectional, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, out_size)
        )

    def forward(self, x, lengths):

        x = self.embedding(x)

        packed_x = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_out, (hidden, cell) = self.lstm(packed_x)

        if self.bidirectional == 2:
            out = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        else:
            out = hidden[-1, :, :]

        x = self.fc(out)
        return x

make_deterministic(seed=42)
model = LSTMClassifier(vocab_size=256, embedding_dim=64, hidden_size=128, num_layers=2, out_size=5, bidirectional=True).to(device)
model

LSTMClassifier(
  (embedding): Embedding(256, 64, padding_idx=0)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=5, bias=True)
  )
)

In [23]:
optimizer = torch.optim.AdamW(model.parameters(), lr = 0.001, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)
valid_loss_fn = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    threshold=1e-3,
)

best_accuracy = 0
best_loss = torch.inf
best_state = None

# Training loop
epochs = 40
for epoch in range(1, epochs+1):
    model.train()
    train_loss = 0.0

    for x, lengths, targets in train_loader:
        x = x.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()

        preds = model(x, lengths)
        loss = loss_fn(preds, targets)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    valid_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, lengths, targets in valid_loader:
            x = x.to(device)
            targets = targets.to(device)

            preds = model(x, lengths)
            loss = valid_loss_fn(preds, targets)
            valid_loss += loss.item()

            predictions = preds.argmax(dim=1)
            correct += (predictions == targets).sum().item()
            total += targets.size(0)

    avg_valid_loss = valid_loss / len(valid_loader)
    avg_train_loss = train_loss / len(train_loader)
    accuracy = (correct / total) * 100

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch: {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | Valid Loss: {avg_valid_loss:.4f} | Valid Acc: {accuracy:.2f}% | LR: {current_lr:.3e}", end="")

    match (accuracy>best_accuracy)-(accuracy<best_accuracy), avg_valid_loss<best_loss:
        case (1, _) | (0, 1):  # compare accuracy first, loss second (since we prioritise accuracy but the metric lacks resolution due to low dataset size)
            best_accuracy = accuracy
            best_loss = avg_valid_loss
            best_state = deepcopy(model.state_dict())
            print("\tNew best state!")
        case _:
            print()
    scheduler.step(avg_valid_loss)

Epoch: 1/40 | Train Loss: 1.5733 | Valid Loss: 1.2251 | Valid Acc: 55.27% | LR: 1.000e-03	New best state!
Epoch: 2/40 | Train Loss: 1.2661 | Valid Loss: 1.0886 | Valid Acc: 64.63% | LR: 1.000e-03	New best state!
Epoch: 3/40 | Train Loss: 1.1259 | Valid Loss: 0.8787 | Valid Acc: 73.64% | LR: 1.000e-03	New best state!
Epoch: 4/40 | Train Loss: 1.0524 | Valid Loss: 0.8852 | Valid Acc: 72.45% | LR: 1.000e-03
Epoch: 5/40 | Train Loss: 0.9634 | Valid Loss: 0.9089 | Valid Acc: 73.64% | LR: 1.000e-03
Epoch: 6/40 | Train Loss: 0.9172 | Valid Loss: 0.8833 | Valid Acc: 76.36% | LR: 1.000e-03	New best state!
Epoch: 7/40 | Train Loss: 0.8136 | Valid Loss: 0.6617 | Valid Acc: 84.35% | LR: 5.000e-04	New best state!
Epoch: 8/40 | Train Loss: 0.7730 | Valid Loss: 0.6404 | Valid Acc: 86.05% | LR: 5.000e-04	New best state!
Epoch: 9/40 | Train Loss: 0.7438 | Valid Loss: 0.6438 | Valid Acc: 86.56% | LR: 5.000e-04	New best state!
Epoch: 10/40 | Train Loss: 0.7176 | Valid Loss: 0.6237 | Valid Acc: 87.76% | L

In [24]:
from datetime import datetime

timestamp = datetime.now().strftime("%d_%m_%H%M")
accuracy_str = f"{best_accuracy:.2f}"
loss_str = f"{best_loss:.3f}"

file_name = f"model_loss_{loss_str}acc_{accuracy_str}_{timestamp}.pth"
save_path = f"/content/drive/MyDrive/sroda_FijalkowskiFilip_ZukowskaRadoslawa/mini_projekt_5/{file_name}"
torch.save(best_state, save_path)

## Dane testowe

In [25]:
best_filename = "model_loss_0.923acc_92.01_23_05_1736.pth"
best_path = f"/content/drive/MyDrive/sroda_FijalkowskiFilip_ZukowskaRadoslawa/mini_projekt_5/{best_filename}"
model.load_state_dict(torch.load(best_path))

<All keys matched successfully>

In [26]:
model.load_state_dict(best_state)

<All keys matched successfully>

In [27]:
import pandas as pd

model.eval()
all_predictions = []

with torch.no_grad():
    for x, lengths in test_loader:
        x = x.to(device)
        preds = model(x, lengths)
        predictions = preds.argmax(dim=1)
        all_predictions.extend(predictions.cpu().tolist())

df = pd.DataFrame(all_predictions)
df.to_csv(f'/content/drive/MyDrive/sroda_FijalkowskiFilip_ZukowskaRadoslawa/mini_projekt_5/pred_{timestamp}.csv', header=False, index=False)

In [28]:
df_preds = pd.DataFrame(all_predictions, columns=['class'])
print(df_preds['class'].value_counts(normalize=True).sort_index()*100)

class
0    54.487761
1    17.679057
2     5.439710
3    14.143246
4     8.250227
Name: proportion, dtype: float64


In [29]:
print(df_labels['class'].value_counts(normalize=True).sort_index()*100)

class
0    55.461041
1    16.264035
2     5.239878
3    15.005104
4     8.029942
Name: proportion, dtype: float64
